# Nemotron **v30 (B)** — RAFT / curriculum train on rollouts → adapter

**Notebook 2 of 2.** Consumes `rollouts.jsonl` from Notebook A and SFTs the **0.85 adapter** on the
kept rollouts. No generation here → fast (~20–40 min), fits 96 GB easily.

## Filter modes (`MODE`)
- **`raft`** — keep the shortest correct rollout(s) from every group with ≥1 correct (uniform weight).
- **`reinforce_rej`** — also drop all-correct groups (too easy → no contrast). Recommended.
- **`curriculum`** (GRPO-flavored, default) — keep correct rollouts, **weight = max(0.1, 1 − pass_rate)**
  → upweight correct answers from HARD groups, downweight easy ones. Reproduces GRPO's
  group-relative-advantage with binary rewards, offline + free.

## Why this is the efficient path
Generation (the bottleneck) already happened in Notebook A via vLLM. Here it's just weighted SFT
(masked-gather CE, per-sequence weight) warm-started from 0.85. Gentle LR (1e-5) = refine, not smash.
Submittable: rank ≤ 32, disk-safe save → `submission.zip`.

## Iterate (approximate on-policy)
After this finishes, point **Notebook A's** `SFT_ADAPTER_DIR` at this notebook's output adapter and
regenerate → run B again. 2–3 rounds ≈ on-policy RAFT. Eval-gate every round vs 0.85.


In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]
target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

In [ ]:
import os, glob

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
MODEL_MAX_LEN = 8192
TRAIN_MAX_LEN = 4096
SEED = 42

def _find(*pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h: return h[0]
    return ""
ROLLOUT_JSONL = _find("/kaggle/input/**/rollouts.jsonl",
                      r"F:/Hackathons/Kaggle-Nemotron/outputs/rollouts.jsonl",
                      "rollouts.jsonl")
SFT_ADAPTER_DIR = "/kaggle/input/models/ramkan07/nemotron-lora-adaptor/pytorch/default/1"
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
RAFT_ADAPTER_DIR = os.path.join(OUT_DIR, "v30_raft_adapter")

# ── RAFT filtering ──
MODE = "curriculum"        # "raft" | "reinforce_rej" | "curriculum"
TOP_K_KEEP = 1             # correct rollouts kept per group (shortest first)
WEIGHT_FLOOR = 0.1         # min per-example weight in curriculum mode

# ── SFT (gentle refine on the 0.85 checkpoint) ──
RAFT_LR = 1e-5             # below from-scratch 2e-4; this is a refine
LR_SCHED = "cosine"
WARMUP_RATIO = 0.05
PER_DEV_BATCH = 1
GRAD_ACCUM = 16            # global batch 16 (small batch = better for LoRA)
NUM_EPOCHS = 1
MAX_GRAD_NORM = 1.0

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

SMOKE = 1
SMOKE_STEPS = 8

print({"jsonl": bool(ROLLOUT_JSONL), "MODE": MODE, "LR": RAFT_LR,
       "batch": PER_DEV_BATCH * GRAD_ACCUM, "SMOKE": SMOKE})


In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys
    def recursive_wheels(pattern):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )
    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")

In [ ]:
# ── warm-start: load 30B + 0.85 adapter as TRAINABLE ──
import os, glob, torch
import kagglehub
from unsloth import FastLanguageModel
from peft import PeftModel

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH, max_seq_length=MODEL_MAX_LEN, load_in_4bit=False,
    full_finetuning=False, trust_remote_code=True, attn_implementation="eager", dtype=torch.bfloat16)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def _resolve(d):
    if d and os.path.exists(os.path.join(d, "adapter_config.json")): return d
    h = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    return os.path.dirname(sorted(h, key=len)[0]) if h else None
A = _resolve(SFT_ADAPTER_DIR)
assert A, "0.85 adapter not found -- set SFT_ADAPTER_DIR"
model = PeftModel.from_pretrained(model, A, is_trainable=True)
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"): model.enable_input_require_grads()
try: model.config.use_cache = False
except Exception: pass
model.train()
model.print_trainable_parameters()
print("warm-started from", A)


In [ ]:
# ── build RAFT/curriculum corpus from rollouts + assistant-masked tokens (+ weight) ──
import json
from datasets import Dataset as HFDataset

SYSTEM_PROMPT = (
    "You solve deterministic logical-puzzle tasks. Infer the exact rule from the examples, apply it "
    "step by step, verify it reproduces the examples, then output the final answer once as \\boxed{...}.")

groups = [json.loads(l) for l in open(ROLLOUT_JSONL, encoding="utf-8") if l.strip()]
recs = []; n_wrong = n_easy = n_mixed = 0
for g in groups:
    rolls = g["rollouts"]; n = len(rolls); nok = sum(r["reward"] > 0 for r in rolls); pr = nok / max(1, n)
    if nok == 0:
        n_wrong += 1; continue
    if pr >= 1.0:
        n_easy += 1
        if MODE in ("reinforce_rej", "curriculum"):
            continue                       # drop too-easy (no contrast)
    else:
        n_mixed += 1
    correct = sorted([r for r in rolls if r["reward"] > 0], key=lambda r: len(r["text"]))[:TOP_K_KEEP]
    w = max(WEIGHT_FLOOR, 1.0 - pr) if MODE == "curriculum" else 1.0
    for c in correct:
        recs.append({"user": g["prompt"] + PROMPT_SUFFIX, "assistant": c["text"].strip(), "weight": float(w)})

print(f"groups: all_wrong={n_wrong} all_correct={n_easy} mixed={n_mixed} -> corpus {len(recs)} (MODE={MODE})")
assert recs, ("empty corpus -- need >=1 correct rollout in non-easy groups. "
              "Lower difficulty / raise G_ROLLOUTS in Notebook A, or set MODE='raft'.")

def _tok_mask(ex):
    full = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ex["user"]},
            {"role": "assistant", "content": ex["assistant"]}]
    def r(m, g):
        try: return tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=g, enable_thinking=True)
        except TypeError: return tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=g)
    fid = tokenizer(r(full, False), add_special_tokens=False, truncation=True, max_length=TRAIN_MAX_LEN)["input_ids"]
    pid = tokenizer(r(full[:2], True), add_special_tokens=False)["input_ids"]
    lab = list(fid)
    for i in range(min(len(pid), len(fid))): lab[i] = -100
    return {"input_ids": fid, "labels": lab, "weight": ex["weight"]}

train_ds = HFDataset.from_list(recs).map(_tok_mask, remove_columns=["user", "assistant"])
train_ds = train_ds.filter(lambda e: any(l != -100 for l in e["labels"]))
if SMOKE:
    train_ds = train_ds.select(range(min(64, len(train_ds))))
print("train rows:", len(train_ds))


In [ ]:
# ── weighted SFT (per-sequence CE x curriculum weight), warm-started from 0.85 ──
import os, time, gc, torch
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments
os.environ["TORCHDYNAMO_DISABLE"] = "1"; os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

class Collate:
    def __init__(self, tok): self.pad = tok.pad_token_id
    def __call__(self, fs):
        m = max(len(f["input_ids"]) for f in fs); ii = []; lb = []; am = []; wt = []
        for f in fs:
            ids = list(f["input_ids"]); la = list(f["labels"]); p = m - len(ids)
            ii.append(ids + [self.pad] * p); lb.append(la + [-100] * p)
            am.append([1] * len(ids) + [0] * p); wt.append(float(f.get("weight", 1.0)))
        return {"input_ids": torch.tensor(ii), "attention_mask": torch.tensor(am),
                "labels": torch.tensor(lb), "weight": torch.tensor(wt, dtype=torch.float)}
collator = Collate(tokenizer)

class WTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        w = inputs.pop("weight"); labels = inputs.pop("labels")
        out = model(**inputs); logits = out.logits
        sl = logits[:, :-1, :]; slb = labels[:, 1:].to(sl.device); mask = (slb != -100)
        V = sl.shape[-1]
        nll = F.cross_entropy(sl.reshape(-1, V).float(), slb.reshape(-1),
                              ignore_index=-100, reduction="none").view(sl.shape[0], -1)
        seq = (nll * mask).sum(1) / mask.sum(1).clamp(min=1)     # per-sequence mean NLL over answer tokens
        loss = (seq * w.to(seq.device)).mean()                   # curriculum weight
        if not torch.isfinite(loss):
            loss = (logits.float().sum() * 0.0).requires_grad_(True)
        return (loss, out) if return_outputs else loss

_total = max(1, len(train_ds) // (PER_DEV_BATCH * GRAD_ACCUM) * NUM_EPOCHS)
_warm = max(1, int(WARMUP_RATIO * _total))
args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "run"), num_train_epochs=NUM_EPOCHS,
    max_steps=SMOKE_STEPS if SMOKE else -1, per_device_train_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=RAFT_LR, lr_scheduler_type=LR_SCHED,
    warmup_steps=_warm, max_grad_norm=MAX_GRAD_NORM, optim="paged_adamw_8bit", bf16=True,
    gradient_checkpointing=False, remove_unused_columns=False, logging_steps=1,
    report_to="none", save_strategy="no", seed=SEED)
print(f"args: total~{_total} warmup={_warm} LR={RAFT_LR} batch={PER_DEV_BATCH*GRAD_ACCUM}")

trainer = WTrainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
t0 = time.time(); trainer.train()
print(f"train done {(time.time()-t0)/60:.1f} min | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB")


In [ ]:
# ── save adapter + submission.zip (disk-safe) ──
import os, json, zipfile, shutil
DEST = RAFT_ADAPTER_DIR; os.makedirs(DEST, exist_ok=True)
trainer.model.save_pretrained(DEST); tokenizer.save_pretrained(DEST)

cfgp = os.path.join(DEST, "adapter_config.json")
cfg = json.load(open(cfgp))
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True; cfg["lora_dropout"] = 0.0
json.dump(cfg, open(cfgp, "w"), indent=2)

_run = os.path.join(OUT_DIR, "run")
if os.path.isdir(_run): shutil.rmtree(_run, ignore_errors=True)   # free disk before zip

need = ["adapter_config.json", "adapter_model.safetensors"]
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else OUT_DIR
if all(os.path.exists(os.path.join(DEST, n)) for n in need):
    z = os.path.join(WORK, "submission.zip")
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        for n in need:
            zf.write(os.path.join(DEST, n), n)
    print(f"submission.zip -> {z}  ({os.path.getsize(z)/1e6:.0f} MB)")
else:
    print("[save] adapter files missing -- re-run the train cell.")

print("\nRAFT round complete. To ITERATE (approx on-policy): set Notebook A's "
      f"SFT_ADAPTER_DIR = '{DEST}', regenerate rollouts, run Notebook B again. Eval-gate vs 0.85.")
